<a href="https://colab.research.google.com/github/Kaunaingul-ai/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kaunaingul-ai/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

### Feature vector

The model uses only public-safe page-level variables that were available in the starter dataset and do not directly reveal the observed outcome.

The selected features are:

- `impressions_90d`
- `ctr`
- `avg_position`
- `content_age_days`
- `days_since_last_update`
- `word_count`

The outcome variable `trend_direction` is not included in the feature vector. The identifier `content_id` is retained only for reference and is also excluded from model inputs.

Because `word_count` contains missing values, I retain those missing values here and handle them later inside the modeling pipeline with an explicit imputation step.

In [6]:
import os
import subprocess
import pandas as pd

REPO_URL = "https://github.com/Kaunaingul-ai/flyrank-ml-internship"
REPO_DIR = "/content/flyrank-ml-internship"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

data_path = os.path.join(
    REPO_DIR,
    "data/raw/content_refresh_anonymized.csv"
)

df = pd.read_csv(data_path)

feature_cols = [
    "impressions_90d",
    "ctr",
    "avg_position",
    "content_age_days",
    "days_since_last_update",
    "word_count"
]

X = df[feature_cols].copy()

print("Feature matrix shape:", X.shape)
print("\nFeature columns:")
print(list(X.columns))

print("\nMissing values:")
print(X.isna().sum())

print("\ntrend_direction included in features:",
      "trend_direction" in X.columns)

print("content_id included in features:",
      "content_id" in X.columns)


Feature matrix shape: (30000, 6)

Feature columns:
['impressions_90d', 'ctr', 'avg_position', 'content_age_days', 'days_since_last_update', 'word_count']

Missing values:
impressions_90d              0
ctr                          0
avg_position                 0
content_age_days             0
days_since_last_update       0
word_count                7699
dtype: int64

trend_direction included in features: False
content_id included in features: False


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

### Feature notes

All six model inputs are numeric page-level variables from the public-safe starter dataset.

- `impressions_90d`: search impressions observed over the available 90-day summary window. No missing values were observed.
- `ctr`: click-through rate associated with the page. No missing values were observed.
- `avg_position`: average search position. No missing values were observed.
- `content_age_days`: approximate age of the content in days. No missing values were observed.
- `days_since_last_update`: time since the content was last updated. No missing values were observed.
- `word_count`: page word count. This field contains missing values and therefore requires explicit handling in the modeling pipeline.

No categorical variables are used in the current feature vector. All selected inputs are treated as information available at scoring time. The observed outcome `trend_direction` is excluded from the feature vector and is used only for evaluation.

In [7]:
feature_notes = pd.DataFrame({
    "feature": feature_cols,
    "dtype": [str(df[c].dtype) for c in feature_cols],
    "missing_n": [int(df[c].isna().sum()) for c in feature_cols],
    "missing_rate": [round(df[c].isna().mean(), 3) for c in feature_cols],
    "categorical": [False] * len(feature_cols),
    "used_as_model_input": [True] * len(feature_cols)
})

print(feature_notes.to_string(index=False))
label_col = "trend_direction"

print("\nAll selected features numeric:",
      all(pd.api.types.is_numeric_dtype(df[c]) for c in feature_cols))

print("Any selected feature equals label column:",
      label_col in feature_cols)

               feature   dtype  missing_n  missing_rate  categorical  used_as_model_input
       impressions_90d   int64          0         0.000        False                 True
                   ctr float64          0         0.000        False                 True
          avg_position float64          0         0.000        False                 True
      content_age_days   int64          0         0.000        False                 True
days_since_last_update   int64          0         0.000        False                 True
            word_count float64       7699         0.257        False                 True

All selected features numeric: True
Any selected feature equals label column: False


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

### Leakage and privacy checks

I explicitly check the selected feature vector for three common problems: outcome leakage, future-window leakage, and private/context fields.

The observed label `trend_direction` must not appear in the model inputs. I also avoid fields that look like post-outcome or future information, and I exclude identifiers or client-specific context from the feature vector.

The purpose of this check is not to prove that every possible form of leakage is impossible, but to document that the selected inputs do not directly contain the known target, identifiers, or obvious future/private fields.

In [8]:
forbidden_exact = {
    "trend_direction",
    "content_id",
    "client_id"
}

future_like_terms = [
    "future",
    "next_",
    "after_",
    "post_",
    "outcome",
    "label"
]

private_like_terms = [
    "client",
    "url",
    "domain",
    "query"
]

exact_forbidden_found = [
    c for c in feature_cols
    if c in forbidden_exact
]

future_like_found = [
    c for c in feature_cols
    if any(term in c.lower() for term in future_like_terms)
]

private_like_found = [
    c for c in feature_cols
    if any(term in c.lower() for term in private_like_terms)
]

print("Leakage/privacy audit")
print("---------------------")
print("Exact forbidden fields in feature vector:", exact_forbidden_found)
print("Future/outcome-like feature names:", future_like_found)
print("Private/context-like feature names:", private_like_found)

print("\nKnown label excluded:",
      "trend_direction" not in feature_cols)

print("Identifiers excluded:",
      all(c not in feature_cols for c in ["content_id", "client_id"]))

Leakage/privacy audit
---------------------
Exact forbidden fields in feature vector: []
Future/outcome-like feature names: []
Private/context-like feature names: []

Known label excluded: True
Identifiers excluded: True


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

### Excluded fields and reasons

I deliberately exclude fields that could create leakage, add unnecessary identifiers, or expose private/context information.

- `trend_direction` — excluded because it is the observed outcome used for evaluation, not a predictor.
- `content_id` — excluded because it is an anonymized identifier and does not describe page performance.
- `client_id` — excluded because it is client/context information and is not needed for the public-safe model.
- Raw URLs, domains, and query text — excluded from the modeling workflow because they may expose private or client-specific information.
- Any future, post-outcome, or label-derived fields — excluded because they would give the model information that would not be available at prediction time.

This keeps the feature vector aligned with the capstone's public-safe and leakage-aware design.

In [9]:
excluded_fields = {
    "trend_direction": "Outcome label; using it as an input would cause target leakage.",
    "content_id": "Identifier only; not a predictive content/search signal.",
    "client_id": "Client/context identifier; excluded from the public-safe feature vector."
}

excluded_table = pd.DataFrame(
    list(excluded_fields.items()),
    columns=["field", "reason"]
)

print(excluded_table.to_string(index=False))

print("\nExcluded fields present in source data:")
for field in excluded_fields:
    print(f"- {field}: {field in df.columns}")

print("\nExcluded fields accidentally used as features:",
      [f for f in excluded_fields if f in feature_cols])

          field                                                                   reason
trend_direction          Outcome label; using it as an input would cause target leakage.
     content_id                 Identifier only; not a predictive content/search signal.
      client_id Client/context identifier; excluded from the public-safe feature vector.

Excluded fields present in source data:
- trend_direction: True
- content_id: True
- client_id: True

Excluded fields accidentally used as features: []


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.